# All layers, three seeds: is the 0.04 gain bigger than training noise?

AllLayers.ipynb found that LoRA on every layer lowers the validation loss by about 0.04 nats against the last 16 layers (0.5B: 3.424 → 3.381, paired -0.043 [-0.068, -0.025]). That interval covers which validation replies were sampled, not training noise: each arm is a single run, seed 0. Here both arms of the 0.5B are trained again with seeds 1 and 2, on the same machine and recipe as those runs (NAS GPU, micro-batch 1, `--grad-accum 4`, gradient checkpointing), so each arm has three seeds.

Seed 0 of each arm is reused: `size_0.5b` (ModelSize.ipynb) and `alllayers_0.5b` (AllLayers.ipynb). Seeds.ipynb measures seed noise too, but on the Mac with micro-batch 4, so it isn't mixed in here.

## Prediction (written before running)

Fede: if training is done right, the variation should be the same for all seeds: they should be very close together. The advantage of all layers is about 0.04, which is what we've seen.

How it's read:
- "Close together": within each arm, the spread of the best validation loss across the three seeds is below 0.02.
- The all-layers gain holds if it is negative for every seed and the mean gain (about 0.04 expected) is larger than the spread within an arm.
- It is refuted if seeds within an arm differ by about as much as the gain (~0.04), or if any seed reverses the sign.

Note written beforehand: seed noise hasn't been measured yet anywhere in this project, so "close together" is a hypothesis, not something seen.

In [1]:
# Heavy steps run scripts through run() and resume: re-running skips what is done. Runs on the NAS:
#   nasrun add bash -c 'export TLM_PY=/opt/venv/bin/python TLM_KEEP_CACHE=1; cd experiments && jupyter nbconvert --execute ... AllLayersSeeds.ipynb'
import json, os, subprocess
from pathlib import Path
import numpy as np
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
PY = os.environ.get("TLM_PY", "uv run python")              # NAS: TLM_PY=/opt/venv/bin/python
MODEL = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
ARMS = {"16 layers": ("size_0.5b", ""), "all layers": ("alllayers_0.5b", "--num-layers -1")}

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line and "█" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

def name(arm, seed):
    base = ARMS[arm][0]
    return base if seed == 0 else f"{base}_seed{seed}"

def best(n):
    v = {int(k): x for k, x in json.load(open(R / f"checkpoints_{n}.json"))["valid"].items()}
    s = min(v, key=v.get)
    return s, v[s]

## Train seeds 1 and 2 of both arms, pick the checkpoint on validation

In [2]:
for seed in [1, 2]:
    for arm, (_, flags) in ARMS.items():
        n = name(arm, seed)
        if not (ROOT / "adapters" / n / "0003500_adapters.safetensors").exists():
            run(f"{PY} scripts/train.py --name {n} --data data/dataset/session --model {MODEL} --iters 3500 --seed {seed} --grad-accum 4 --grad-checkpoint {flags}")
        run(f"{PY} scripts/eval_checkpoints.py adapters/{n} --model {MODEL} --data data/dataset/session/valid.jsonl --steps 500:3500:500 --out experiments/results/checkpoints_{n}.json")

$ /opt/venv/bin/python scripts/train.py --name size_0.5b_seed1 --data data/dataset/session --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --iters 3500 --seed 1 --grad-accum 4 --grad-checkpoint
5404 train examples, batch 4: 3500 steps (2.59 epochs)
/opt/venv/bin/python -m mlx_lm lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --train --data data/dataset/session --mask-prompt --adapter-path /work/talk-like-me/.nasrun/snap/alllayers-seeds/adapters/size_0.5b_seed1 --iters 14000 --batch-size 1 --learning-rate 1e-05 --num-layers 16 --max-seq-length 2048 --save-every 400 --steps-per-eval 400 --val-batches 25 --seed 1 --grad-accumulation-steps 4 --grad-checkpoint
Loading pretrained model
Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    model           mlx-community/Qwen2.5-0.5B-Instruct-4bit                                                          │
│    type            lora · 16 laye

## Summary

In [3]:
rows = {arm: [best(name(arm, s)) for s in [0, 1, 2]] for arm in ARMS}
print(f"{'arm':<11} {'seed 0':>14} {'seed 1':>14} {'seed 2':>14} {'mean':>7} {'sd':>6}")
for arm, r in rows.items():
    losses = [l for _, l in r]
    print(f"{arm:<11} " + " ".join(f"{l:.3f} @{s:<5}" for s, l in r) + f" {np.mean(losses):>7.3f} {np.std(losses, ddof=1):>6.3f}")
diffs = [rows["all layers"][i][1] - rows["16 layers"][i][1] for i in range(3)]
print()
print("all - 16 by seed: " + ", ".join(f"{d:+.3f}" for d in diffs) + f" | mean {np.mean(diffs):+.3f}")
print(f"largest seed-to-seed spread within an arm: {max(max(l for _, l in r) - min(l for _, l in r) for r in rows.values()):.3f}")

arm                 seed 0         seed 1         seed 2    mean     sd
16 layers   3.424 @2500  3.440 @2000  3.417 @2500    3.427  0.012
all layers  3.381 @2500  3.403 @2000  3.381 @2500    3.389  0.013
all - 16 by seed: -0.043, -0.037, -0.036 | mean -0.038
largest seed-to-seed spread within an arm: 0.023


## Conclusion

**Prediction (Fede):** seeds land very close together, and the ~0.04 advantage of all layers holds.

**Right on what matters, borderline on the detail.**
- **The all-layers gain repeats in every seed:** -0.043, -0.037, -0.036 nats (mean -0.038). Same sign, nearly the same size, three times. AllLayers.ipynb's result is not a lucky seed.
- **Seeds are close, but not quite inside the 0.02 written beforehand:** best validation loss moves by up to 0.023 within an arm (16 layers 3.417-3.440, sd 0.012; all layers 3.381-3.403, sd 0.013). That spread is about half the gain, so the gain is larger than training noise.
- **The seed effect is shared by both arms:** seed 1 is the worst seed for both (3.440 and 3.403) and also picks step 2,000 in both. Whatever a seed does (LoRA initialisation, data order) moves both arms together, so the paired gain is much steadier (-0.036 to -0.043) than either arm alone.

**What this sets for the rest of the project:** on the NAS recipe, about ±0.01 (sd) of the best validation loss is training noise for one 0.5B run. Differences between single runs below ~0.02-0.025 shouldn't be read as real; LoraRank.ipynb's spread across ranks (3.398-3.422) falls inside that band, which supports "rank doesn't matter". Seeds.ipynb measures the same for the Mac recipe.
